# 05 — Federated linear regression

A toy FedAvg-style example: `N_WORKERS` each hold their own partition of a
synthetic dataset for `y = w*x + b`. Each round, every worker computes the
gradient of the squared-error loss on its own partition (at the *current*
shared `w`/`b`), posts it via `Federator.Map`, and the head aggregates with
`SUM`, averages across workers, and takes one gradient-descent step. Because
each worker posts under the same fixed key every round, next round's `Map`
call simply overwrites this round's value - no explicit reducer reset is
needed between rounds here.

In [ ]:
import os
import numpy as np
import pandas as pd
from scarlets.core.Mapper import Mapper
from scarlets.formulations.Federator import Federator
from scarlets.utils.ScarletUtils import redisConnect

os.environ.setdefault("REDIS_HOST", "localhost")
os.environ.setdefault("REDIS_PORT", "6379")
os.environ.setdefault("REDIS_AUTH_TOKEN", "")
os.environ.setdefault("APP_ID", "notebook_worker")

REG_NAME = "tutorial_regression"

## Cleanup

In [ ]:
def cleanup():
    r = redisConnect()
    for base in (f"{REG_NAME}_mapper_reducer", f"{REG_NAME}_mapper_global"):
        for pattern in (f"{base}_key-value:*", f"{base}_key-list"):
            keys = list(r.scan_iter(match=pattern))
            if keys:
                r.delete(*keys)

cleanup()
print("cleaned up")

## Synthetic data, partitioned across workers

True line: `y = 3.0*x - 1.0`, plus noise. Split into `N_WORKERS` disjoint
partitions - each worker only ever sees its own slice.

In [ ]:
rng = np.random.default_rng(seed=0)

TRUE_W, TRUE_B = 3.0, -1.0
N_WORKERS = 4
N_POINTS_PER_WORKER = 50

X = rng.uniform(-5, 5, size=(N_WORKERS, N_POINTS_PER_WORKER))
Y = TRUE_W * X + TRUE_B + rng.normal(scale=1.0, size=X.shape)

## Local gradient

Standard mean-squared-error gradient, evaluated on one worker's partition
at the current shared `(w, b)`.

In [ ]:
def local_gradient(x, y, w, b):
    pred = w * x + b
    err = pred - y
    grad_w = np.mean(2 * err * x)
    grad_b = np.mean(2 * err)
    return np.array([grad_w, grad_b])

## Federated training loop

Each round: every worker computes its local gradient and posts it; the head
aggregates (sum) and divides by `N_WORKERS` to average; one gradient step
is taken on the shared `(w, b)`.

In [ ]:
LR = 0.03
N_ROUNDS = 60

w, b = 0.0, 0.0
history = []

for round_i in range(N_ROUNDS):
    for worker_id in range(N_WORKERS):
        fed = Federator(REG_NAME, op=Mapper.SUM)
        grad = local_gradient(X[worker_id], Y[worker_id], w, b)
        fed.Map(grad, key=f"worker_{worker_id}")

    fed_head = Federator(REG_NAME, op=Mapper.SUM)
    total_grad, ok, exc = fed_head.Aggregate(np.zeros(2))
    avg_grad = total_grad / N_WORKERS

    w -= LR * avg_grad[0]
    b -= LR * avg_grad[1]
    history.append({"round": round_i, "w": w, "b": b})

    if round_i % 5 == 0 or round_i == N_ROUNDS - 1:
        print(f"round {round_i:2d}: w={w:.3f}  b={b:.3f}")

print(f"\ntrue:    w={TRUE_W}  b={TRUE_B}")
print(f"learned: w={w:.3f}  b={b:.3f}")

## Convergence

In [ ]:
pd.DataFrame(history).set_index("round").plot(title="Federated gradient descent", ylabel="parameter value");

## Cleanup (teardown)

In [ ]:
cleanup()
print("cleaned up")